<a href="https://colab.research.google.com/github/Yousra-khallou/-NLP-Tasks/blob/main/TP2_EDM_Autocorrection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **TP N°2 : L’algorithme Minimum Edit Distance et Autocorrection**


In [1]:
!pip install jellyfish

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 9.0 MB/s eta 0:00:00


In [2]:
import numpy as np
import ast
import jellyfish
import string

# Exercice 1

Implémentation d'algorithme de distance d'édition minimale.

In [3]:
#distance cost functions
def del_cost(S_caracter):
  return 1
def ins_cost(T_caracter):
  return 1
def sub_cost(caracter1, caracter2):
  return 0 if caracter1==caracter2 else 2

In [11]:
def Min_Edit_Distance(source, target):

  n=len(source)
  m=len(target)
  D_matrix = [[0] * (m + 1) for _ in range(n+1)]  #matrice de taille (n+1 , m+1)
  # Initialization: the zeroth row and column is the distance from the empty string
  D_matrix[0][0]=0
  for i in range(1,n+1):
      D_matrix[i][0] = D_matrix[i-1][0]+ del_cost(source[i-1])
  for j in range(1,m+1):
    D_matrix[0][j] = D_matrix[0][j-1]+ ins_cost(target[j-1])
  # Recurrence relation:
  for i in range(1,n+1):
    for j in range(1,m+1):
      D_matrix[i][j]= min(D_matrix[i-1][j] + del_cost(source[i-1]) , D_matrix[i-1][j-1] + sub_cost(source[i-1],target[j-1]) , D_matrix[i][j-1] + ins_cost(target[j-1]))


  # Termination
  return D_matrix


Test de Min_Edit_Distance(source, target)

In [12]:

source = "ab"
target = "cab"

D = Min_Edit_Distance(source, target)

# Afficher la matrice
print("Matrice des distances :")
for row in D:
    print(row)

# Afficher la distance finale
print("\nDistance minimale d'édition :", D[len(source)][len(target)])

Matrice des distances :
[0, 1, 2, 3]
[1, 2, 1, 2]
[2, 3, 2, 1]

Distance minimale d'édition : 1


Enrichissement de la fonction Min_Edit_Distance pour enregistrer le chemin de retour en arrière (backtrace)

In [26]:
def med_with_backtrace(source, target):
  n=len(source)
  m=len(target)
  D_matrix = [[0] * (m + 1) for _ in range(n+1)]
  pointers= [[0] * (m + 1) for _ in range(n+1)]

  # Initialization: the zeroth row and column is the distance from the empty string
  D_matrix[0][0]=0

  for i in range(1,n+1):
      D_matrix[i][0] = D_matrix[i-1][0]+ del_cost(source[i-1])
      pointers[i][0]="←"
  for j in range(1,m+1):
      D_matrix[0][j] = D_matrix[0][j-1]+ ins_cost(target[j-1])
      pointers[0][j] ="↑"

  # Recurrence relation:
  for i in range(1,n+1):
      for j in range(1,m+1):

          del_=D_matrix[i-1][j] + del_cost(source[i-1])
          ins_=D_matrix[i][j-1] + ins_cost(target[j-1])
          sub_=D_matrix[i-1][j-1] + sub_cost(source[i-1],target[j-1])

          Min = min(del_, ins_, sub_)
          D_matrix[i][j]= Min
          if del_ == ins_ == sub_:
              pointers[i][j] = "↑ ← ↖"
          elif del_ == ins_:
              pointers[i][j] = "↑ ←"
          elif del_ == sub_:
              pointers[i][j] = "↑ ↖"
          elif ins_ == sub_:
              pointers[i][j] = "← ↖"
          elif Min == del_:
              pointers[i][j] = "↑"
          elif Min == ins_:
              pointers[i][j] = "←"
          else:
              pointers[i][j] = "↖"


  # Termination with backtrace_matrix
  return D_matrix, pointers


In [27]:

source = "ab"
target = "cab"

D,P = med_with_backtrace(source, target)
# Afficher la matrice
print("Matrice des distances :")
for row in D:
    print(row)
print("matrice des pointeurs :")
for row in P:
    print(row)


# Afficher la distance finale
print("\nDistance minimale d'édition :", D[len(source)][len(target)])

Matrice des distances :
[0, 1, 2, 3]
[1, 2, 1, 2]
[2, 3, 2, 1]
matrice des pointeurs :
[0, '↑', '↑', '↑']
['←', '↑ ← ↖', '↑ ←', '↑ ↖']
['←', '↑ ← ↖', '← ↖', '↑ ←']

Distance minimale d'édition : 1


# première version d’un correcteur d’orthographe

Soit w un mot contenant une erreur d’orthographe (un mot absent du dictionnaire), calculer la
distance entre w et tous les mots du dictionnaire contenant les mots valides puis proposer les k
corrections possibles (les k plus proches mots selon la distance d'édition minimale).  
Vous pouvez utiliser le dictionnaire français donnée dans le fichier dictionnaire_fr.txt.

In [28]:
def compute_distances_dictionary(word):

    with open("/content/drive/MyDrive/dictionnaire_fr.txt", "r", encoding="utf-8") as f:
        dictionnaire = f.read().splitlines()

    distances_dictionary = np.empty((2, len(dictionnaire)+1), dtype=object)
    distances_dictionary[1][0] = word  # mot source en [1][0]

# Copier les mots du dictionnaire à partir de la colonne 1
    for j in range(1, len(dictionnaire)+1):
        distances_dictionary[0][j] = dictionnaire[j-1]

    m = distances_dictionary.shape[1]

    for j in range(1, m+1):
        distances_dictionary[1][j] = Min_Edit_Distance_smart(distances_dictionary[1][0], distances_dictionary[0][j])[len(word)][len(distances_dictionary[0][j])]

    return distances_dictionary


In [29]:
def find_closest_words(distance_matrix, k):

    # Remplacer None par inf pour le tri
    distances = np.array([
        d if d is not None else np.inf
        for d in distance_matrix[1, 1:]
    ], dtype=float)

    indices = np.argsort(distances)

    distance_matrix_sorted = distance_matrix[:, np.concatenate(([0], indices + 1))]

    return distance_matrix_sorted[:, :k+1]

Optimisation l’algorithme précédent

In [ ]:
def compute_distances_dictionary_opt(word):

    with open("/content/drive/MyDrive/dictionnaire_fr.txt", "r", encoding="utf-8") as f:
        dictionnaire = f.read().splitlines()

    # Filtrer les mots avec même première lettre et même longueur
    dictionnaire = [mot for mot in dictionnaire
                    if mot[0].lower() == word[0].lower()
                    and len(mot) == len(word)]

    distances_dictionary = np.empty((2, len(dictionnaire)+1), dtype=object)
    distances_dictionary[1][0] = word

    for j in range(1, len(dictionnaire)+1):
        distances_dictionary[0][j] = dictionnaire[j-1]

    m = distances_dictionary.shape[1]

    for j in range(1, m):
        distances_dictionary[1][j] = Min_Edit_Distance_smart(distances_dictionary[1][0], distances_dictionary[0][j])[len(word)][len(distances_dictionary[0][j])]

    return distances_dictionary

In [ ]:
def compute_distances_dictionary(word):

    with open("/content/drive/MyDrive/dictionnaire_fr.txt", "r", encoding="utf-8") as f:
        dictionnaire = f.read().splitlines()

    n = len(word)
    word_lower = word.lower()
    word_soundex = jellyfish.soundex(word)

    # meme première lettre (insensible à la casse)
    dictionnaire = [mot for mot in dictionnaire
                    if mot[0].lower() == word_lower[0]]

    # Longueur similaire plus ou moins de 2 caracteres pour une large recherche
    dictionnaire = [mot for mot in dictionnaire
                    if abs(len(mot) - n) <= 2]

    # 3. Même code Soundex (même son)
    dictionnaire = [mot for mot in dictionnaire
                    if jellyfish.soundex(mot) == word_soundex]


    if not dictionnaire:
        return None  # aucun mot candidat trouvé

    distances_dictionary = np.empty((2, len(dictionnaire)+1), dtype=object)
    distances_dictionary[1][0] = word

    for j in range(1, len(dictionnaire)+1):
        distances_dictionary[0][j] = dictionnaire[j-1]

    m = distances_dictionary.shape[1]

    SEUIL = 10  # early stopping
    for j in range(1, m):
        mot = distances_dictionary[0][j]
        dist = Min_Edit_Distance_smart(word, mot)[n][len(mot)]
        distances_dictionary[1][j] = dist if dist <= SEUIL else None

    return distances_dictionary

In [ ]:
# Debug : afficher le contenu brut
distances = compute_distances_dictionary_opt('bonjour')
print(distances)
print(distances[1, 1:])

[[None 'baccara' 'bagages' 'bagarre' 'Bahamas' 'Bahrein' 'baigner'
  'bailler' 'baisser' 'baklava' 'balader' 'balafre' 'balance' 'balatum'
  'balayer' 'baleine' 'Balkans' 'ballade' 'ballast' 'ballets' 'ballons'
  'balourd' 'bambins' 'bandage' 'Bangkok' 'banques' 'banquet' 'bapteme'
  'baptise' 'baraque' 'baraque' 'Barbara' 'barbare' 'barbier' 'bariole'
  'baroque' 'barrage' 'barreau' 'baryton' 'basalte' 'bascule' 'bascule'
  'basique' 'bassine' 'bastion' 'Batavia' 'bateaux' 'batisse' 'battage'
  'battant' 'battent' 'batteur' 'bavarde' 'Baviere' 'bavures' 'Bayonne'
  'Beatles' 'Beauvau' 'Beckett' 'becoter' 'begayer' 'bel age' 'Belarus'
  'Belfast' 'Belfond' 'Belfort' 'bercail' 'berceau' 'Bergman' 'Berlioz'
  'Bernard' 'besogne' 'besoins' 'beurrer' 'biaiser' 'bibelot' 'bientot'
  'bigarre' 'bigleux' 'billets' 'binaire' 'Birmane' 'Birmans' 'biscuit'
  'bistrot' 'bitumer' 'biturer' 'bivouac' 'bizarre' 'blafard' 'blaguer'
  'blanche' 'blessee' 'blesser' 'blesses' 'blindes' 'blocage' 'bloque

In [ ]:
# ---- TESTS ----
mots_a_tester = ["bonjour", "ecritre", "langaje", "informtion", "phonne"]

for mot in mots_a_tester:
    print("-" * 40)
    print(f"Mot saisi : '{mot}'")

    distances = compute_distances_dictionary(mot)

    if distances is None:
        print("Aucun candidat trouvé")
        continue

    top_k = find_closest_words(distances, k=5)

    print("Top 5 corrections :")
    for j in range(1, top_k.shape[1]):
        mot_candidat = top_k[0][j]
        dist         = top_k[1][j]
        print(f"  {j}. '{mot_candidat}'    distance = {dist}")

print("-" * 40)

----------------------------------------
Mot saisi : 'bonjour'


TypeError: '<' not supported between instances of 'NoneType' and 'NoneType'

# Exercice 2


Implémenter la fonction process_data(corpus_file) qui permet de lire le corpus donné sous
forme d’un fichier texte, convertir le texte en minuscule et segmenter le texte et retourne la liste
des mots.

In [ ]:
def process_data(path_fichier):
    with open(path_fichier, "r", encoding="utf-8") as f:
        contenu = f.read().lower()
        contenu = contenu.translate(str.maketrans("", "", string.punctuation))
        mots = contenu.split()
    return mots



mots=process_data("/content/drive/MyDrive/big.txt")
print(mots[:10])


['the', 'project', 'gutenberg', 'ebook', 'of', 'the', 'adventures', 'of', 'sherlock', 'holmes']


Implémenter la fonction get_vocabulary(corpus_file) qui retourne le vocabulaire construit à
partir d’un corpus passé en argument de la fonction.

In [ ]:
def get_vocabulary(path_file):

  vocabulary = list(set(process_data(path_file)))
  return vocabulary

print(get_vocabulary("/content/drive/MyDrive/big.txt")[:5])

['missouri', 'waves', 'jowl', 'lookingglass', 'exhaustion']


3-Nous pouvons estimer la probabilité d’un mot, en comptant le nombre de fois où ce mot
apparaît dans un corpus de texte de grande taille et en divisant sur la taille de corpus. Ecrire
une fonction qui construit le modèle de langue en calculant la probabilité de chaque mot en se
basant sur le fichier big.txt fournit avec ce TP et elle stocke le résultat dans une structure de
données adéquate. N’oubliez pas d’effectuer le preprocessing avec la fonction process_data.  

In [ ]:
# get_count pour Compte le nombre d'occurrences de chaque mot

def get_count(mots):  # mots : list des mots du corpus
    count = {}        # dictionnare mot: count occurrence
    for mot in mots:
        count[mot] = count.get(mot, 0) + 1
    return count

# Calcule la probabilité P(c) de chaque mot

def get_proba(mots):
    count=get_count(mots)
    total = sum(count.values())  # nombre total de mots en sommant count de tout les mots
    proba_dict = {}
    for mot, count_mot in count.items():
        proba_dict[mot] = count_mot / total
    return proba_dict



4-Ecrire les fonctions suivantes :
 - - -
edits1(s) :   qui retourne l’ensemble de toutes les chaînes (qu'il s'agisse de mots ou non)
qui peuvent être obtenues avec une seule modification (insertion, substitution ou
suppression) à effectuer sur la chaîne s.

edits2(s) :   qui retourne l’ensemble de toutes les chaînes (qu'il s'agisse de mots ou non)
qui peuvent être obtenues avec 2 modifications (insertion, substitution ou suppression) à
effectuer sur la chaîne s.

knownWord(words) qui filtre les mots de la liste words qui ne sont pas dans le vocabulaire
(elle garde donc que les mots valides). On peut utiliser la fonction get_vocabulary(corpus_file)

In [ ]:
def edits1(s):

    lettres = 'abcdefghijklmnopqrstuvwxyz'

    # Toutes les façons de diviser s en deux parties (gauche, droite)
    splits = [(s[:i], s[i:]) for i in range(len(s) + 1)]

    # supprimer un caractère
    suppressions = [g + d[1:] for g, d in splits if d]

    # remplacer un caractère par une lettre
    substitutions = [g + l + d[1:] for g, d in splits if d for l in lettres]

    # insérer une lettre
    insertions = [g + l + d for g, d in splits for l in lettres]

    return list(set(suppressions + substitutions + insertions))

In [ ]:
# appliqué 2 fois edits1 , 1 ere fois sur s 2ème fois sur les mots obtenus
def edits2(s):
    return list(set(e2 for e1 in edits1(s) for e2 in edits1(e1)))

In [ ]:
#Filtre les mots qui existent dans le vocabulaire.
def known_words(words, corpus_file):

    vocabulaire = get_vocabulary(corpus_file)
    return list(set(w for w in words if w in vocabulaire))

On suppose que nous n’avons pas des données pour construire le modèle d’erreur, ainsi nous allons adopter les hypothèses suivantes:
tous les mots connus avec une distance d'édition de 1 sont infiniment plus probables que les mots connus avec une distance d'édition de 2,
et infiniment moins probables qu'un mot connu avec une distance d'édition de 0.   
Ainsi, pour sélectionner les candidats les plus probables nous considérons leurs probabilités selon le modèle de langue préalablement construit et leurs priorités en fonction de leur distance d’édition du mot original.   Avec cette simplification, nous n'avons pas besoin de multiplier par
un facteur P(w|c), car chaque candidat à la priorité choisie aura la même probabilité.

Ecrire la fonction **candidates(word)** qui retourne  la première liste non vide de candidats par ordre de priorité :
Le mot original, s'il est connu ; sinon
La liste des mots connus à une distance d'édition de un, s'il y en a ; sinon
La liste des mots connus à une distance d'édition de deux, s'il y en a ; sinon
Le mot original, même s'il n'est pas connu.

In [ ]:
def candidates(word, corpus_file):
    mots        = process_data(corpus_file)
    vocabulaire = get_vocabulary(corpus_file)
    proba       = get_proba(mots)

    # distance 0 : mot déjà correct
    mot_correct = known_words([word], corpus_file)

    # distance 1 : mots à 1 modification, triés par probabilité
    mots_edits1 = known_words(edits1(word), corpus_file)        # fix: word
    mots_edits1_triee = dict(sorted(
                            {m: proba.get(m, 0) for m in mots_edits1}.items(),
                            key=lambda x: x[1], reverse=True))

    # distance 2 : mots à 2 modifications, triés par probabilité
    mots_edits2 = known_words(edits2(word), corpus_file)        # fix: edits2
    mots_edits2_triee = dict(sorted(
                            {m: proba.get(m, 0) for m in mots_edits2}.items(),
                            key=lambda x: x[1], reverse=True))

    # retourner par ordre de priorité
    if mot_correct:
        return mot_correct
    if mots_edits1:
        return mots_edits1_triee                                # fix: trié
    if mots_edits2:
        return mots_edits2_triee                                # fix: trié
    return [word]                                               # fix: mot inconnu

Test


In [ ]:
word = "lookingglasa"
resultat = candidates(word, "/content/drive/MyDrive/big.txt")
print(f"Test 1 - mot correct")
print(f"  mot      : {word}")
print(f"  candidats: {resultat}")
print()

Test 1 - mot correct
  mot      : lookingglasa
  candidats: {'lookingglass': 9.133237495227884e-07}



In [ ]:
word = "thhe"
resultat = candidates(word, "/content/drive/MyDrive/big.txt")
print(f"Test 2 - distance 1")
print(f"  mot      : {word}")
print(f"  candidats: {resultat}")
print()

Test 2 - distance 1
  mot      : thhe
  candidats: {'the': 0.07250785915086465, 'thee': 2.3746417487592497e-05}



In [ ]:
word = "thhee"
resultat = candidates(word, "/content/drive/MyDrive/big.txt")
print(f"Test 3 - distance 2")
print(f"  mot      : {word}")
print(f"  candidats: {resultat}")
print()

Test 3 - distance 2
  mot      : thhee
  candidats: {'three': 0.00047949496849946385, 'thee': 2.3746417487592497e-05, 'thwee': 2.739971248568365e-06}



In [ ]:
word = "xzqrt"
resultat = candidates(word, "/content/drive/MyDrive/big.txt")
print(f"Test 4 - mot inconnu")
print(f"  mot      : {word}")
print(f"  candidats: {resultat}")
print()

Test 4 - mot inconnu
  mot      : xzqrt
  candidats: ['xzqrt']

